In [ ]:
import numpy as np
import torch
from sbi import DeepSets, train, evaluate, compare_models, BetaProposal

# Problem config
mu0, sigma0 = 0, 1
A = 5
n = 64

In [ ]:
def sample_batch(batch_size):
    mu = np.random.normal(mu0, sigma0, size=batch_size)
    sigma = np.random.uniform(0, A, size=batch_size)
    x = np.array([np.random.normal(mu[i], sigma[i], size=n) for i in range(batch_size)])
    theta = np.stack([mu, sigma], axis=1)
    return theta, x

class DeepSetsNormal(DeepSets):
    def __init__(self):
        super().__init__(input_dim=2, output_dim=2)

    def alter_inputs(self, x):
        var = torch.var(x, dim=1, keepdim=True)
        log_var = torch.log(var).expand_as(x)
        return torch.stack([x, log_var], dim=-1)

    def alter_outputs(self, x):
        mu = x[:, 0]
        sigma = torch.sigmoid(x[:, 1]) * A
        return torch.stack([mu, sigma], dim=1)

In [ ]:
model_base = DeepSetsNormal()
model_base, history_base = train(model_base, sample_batch, n_epochs=100)
evaluate(model_base, sample_batch, history_base, [r'$\mu$', r'$\sigma$'])

In [ ]:
proposal = BetaProposal([
    {'name': 'mu',    'lo': -4, 'hi': 4, 'alpha': 1.0, 'beta': 1.0},  # mu stays ~uniform
    {'name': 'sigma', 'lo': 0,  'hi': A, 'alpha': 0.5, 'beta': 1.0},  # oversample small sigma
])

def sample_batch_is(batch_size, alpha_sigma=0.5):
    # Use proposal for sigma, normal prior for mu
    mu = np.random.normal(mu0, sigma0, size=batch_size)
    sigma_tilde = np.random.beta(alpha_sigma, 1.0, size=batch_size)
    sigma = A * sigma_tilde
    x = np.array([np.random.normal(mu[i], sigma[i], size=n) for i in range(batch_size)])
    from scipy import stats
    q = stats.beta.pdf(sigma_tilde, alpha_sigma, 1.0)
    weights = 1.0 / q  # p(sigma)/q(sigma) = (1/A) / (Beta_pdf/A)
    theta = np.stack([mu, sigma], axis=1)
    return theta, x, weights

In [ ]:
models = {'Baseline': model_base}
histories = {'Baseline': history_base}

for alpha in [0.7, 0.5, 0.3]:
    name = f'IS a={alpha}'
    fn = lambda bs, a=alpha: sample_batch_is(bs, alpha_sigma=a)
    m = DeepSetsNormal()
    m, h = train(m, fn, n_epochs=100, seed=0)
    models[name] = m
    histories[name] = h

In [ ]:
test_set = sample_batch(4000)
test_set = (torch.tensor(test_set[0], dtype=torch.float32),
            torch.tensor(test_set[1], dtype=torch.float32))
compare_models(models, histories, test_set, [r'$\mu$', r'$\sigma$'],
               param_ranges=[(-4, 4), (0, A)])